In [ ]:
from pathlib import Path
import pandas as pd
from typing import List, Sequence, Optional, Any
from plotnine import (
    ggplot, aes, geom_line, geom_point,
    ggtitle, xlab, ylab, theme,
    element_text
)
from plotnine.themes import theme_classic  # Cleaner theme for publication
from plotnine.scales import scale_color_hue # For distinct colors

In [ ]:
# --- Load Excel file ---
def load_file(file_name: str, sheet_name: Optional[str] = None) -> pd.DataFrame:
    """Load an Excel file as a DataFrame with a fast engine for Excel files.
    Parameters:
    - file_path: path to Excel file
    - sheet_name: sheet name or None to load the first sheet
    Returns a pandas DataFrame."""
    # 1. Get the folder where THIS script lives
    script_location = Path.cwd()
    # 2. Build the path to the file relative to the script
    file_path = script_location / "Data" / file_name
    
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {file_path}')
    # Let pandas choose a working engine (openpyxl, xlrd, etc.). If your environment requires a specific engine, pass it here.
    return pd.read_excel(path, sheet_name=sheet_name, engine="calamine")

# Example (change to your file or pass via the main() entry below):
df = load_file("Moha_LFP_5C_14-3mg_additive_Channel_12_Wb_1.xlsx", "Channel-12_1")

In [3]:
# --- Filtering function ---
def filtering(df: pd.DataFrame, filter_column_names: Sequence[str], ranges: Sequence[Sequence[Any]]) -> pd.DataFrame:
    """Filter `df` by multiple columns using membership tests (isin).
    - `filter_column_names`: sequence of column names to filter on
    - `ranges`: sequence of sequences containing allowed values for each corresponding column
    Returns the filtered DataFrame."""
    if len(filter_column_names) != len(ranges):
        raise ValueError('filter_column_names and ranges must have the same length')
    filtered_df = df.copy()
    for col, allowed in zip(filter_column_names, ranges):
        if col not in filtered_df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame")
        filtered_df = filtered_df[filtered_df[col].isin(allowed)]
    return filtered_df

In [5]:
def plotnine_ploting(
    df: pd.DataFrame,
    multiple_graphs: bool = True,
    legends: bool = True,
    points_and_lines: int = 1,
    column_filter_name: Optional[str] = 'Cycle_Index',
    column_X: str = 'Cycle_Index',
    column_Y: str = 'Specific Capacity (mAh/g)',
    plot_title: str = 'Plot',
    folderName: str = 'default',
    markevery: int = 50,
    save_folder_base: str = 'Cycling/plots',
    overwrite: bool = False,
    dpi: int = 400,
    width: float = 8,
    height: float = 6,
    units: str = 'in'):
    # Prepare grouping column if present
    if column_filter_name and column_filter_name in df.columns:
        df[column_filter_name] = df[column_filter_name].astype('category')

    # Build ggplot using aes_string for dynamic column names
    if multiple_graphs and column_filter_name:
        gg = ggplot(df, aes(x=column_X, y=column_Y, color=column_filter_name, group=column_filter_name))
    else:
        gg = ggplot(df, aes(x=column_X, y=column_Y))

    # Sample points for markers
    if markevery <= 1:
        point_df = df.copy()
    else:
        if multiple_graphs and column_filter_name and column_filter_name in df.columns:
            point_df = df.groupby(column_filter_name, group_keys=False, observed=True).apply(lambda x: x.iloc[::markevery]).reset_index(drop=True)
        else:
            point_df = df.iloc[::markevery].copy()

    # Add layers based on points_and_lines flag: 2=line+points, 1=points, 0=line only
    if points_and_lines == 2:
        gg = gg + geom_line(size=0.5, linetype='solid') + geom_point(data=point_df, size=3)
    elif points_and_lines == 1:
        gg = gg + geom_point(data=point_df, size=3)
    else:
        gg = gg + geom_line(size=0.5, linetype='solid')

    if multiple_graphs:
        gg = gg + scale_color_hue(l=0.4, s=0.8)

    gg = (gg + ggtitle(plot_title)
          + xlab(column_X)
          + ylab(column_Y)
          + theme_classic(base_size=14)
          + theme(
              plot_title=element_text(weight='bold', size=14),
              axis_title=element_text(weight='bold', size=14),
              legend_title=element_text(weight='bold', size=12),
              legend_position='right'
          ))

    if not legends:
        gg = gg + theme(legend_position='none')

    # Saving paths
    save_folder = Path(save_folder_base) / folderName
    save_folder.mkdir(parents=True, exist_ok=True)
    safe_title = plot_title.replace(' ', '_').replace('/', '_')
    save_path_png = str(save_folder / (safe_title + '.png'))
    save_path_pdf = str(save_folder / (safe_title + '.pdf'))

    # Save PNG
    if Path(save_path_png).exists() and not overwrite:
        print(f"Skipping existing PNG (overwrite=False): {save_path_png}")
    else:
        gg.save(save_path_png, dpi=dpi, width=width, height=height, units=units)
        print(f"PNG saved to {save_path_png}")

    # Save PDF
    if Path(save_path_pdf).exists() and not overwrite:
        print(f"Skipping existing PDF (overwrite=False): {save_path_pdf}")
    else:
        gg.save(save_path_pdf, width=width, height=height, units=units)
        print(f"PDF saved to {save_path_pdf}")

    return save_path_png, save_path_pdf

In [14]:
# --- Main Execution Block / CLI entry point ---
def build_selected_cycles(df: pd.DataFrame) -> List[int]:
    cycles = sorted(df['Cycle_Index'].unique())
    return sorted(set(cycles[2:3] + [c for c in cycles if c % 100 == 0] + cycles[-3:-1]))
#sorted(set(cycles[:2] + [c for c in cycles[:-800] if c % 10 == 0]))
#sorted(set(cycles[:2] + [c for c in cycles if c % 10 == 0] + cycles[-800:-800]))

#df = load_file(r'C:/Users/MTP24ME/Documents/CODE_PHD/meizeddin_PhD_code/Cycling/Data/Moha_NMC_200_73_Channel_12_Wb_1.xlsx', 'Channel-12_1')
selected_cycles = build_selected_cycles(df)
print('Selected cycles:', selected_cycles)

# 1. Discharge retention (example)
filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[3, 7], [2.5], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, legends=True, 
                 points_and_lines=1, column_filter_name='Cycle_Index', 
                 column_X='Cycle_Index', column_Y='Specific Capacity (mAh/g)', 
                 plot_title='Discharge Retention', folderName='LFP_5C_14-3mg_additive', 
                 save_folder_base='Cycling/plots', overwrite=True)

# 2. Charge retention (example)
filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[2, 6], [3.7], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, legends=True, 
                 points_and_lines=1, column_filter_name='Cycle_Index', 
                 column_X='Cycle_Index', column_Y='Specific Capacity (mAh/g)', 
                 plot_title='Charge Retention', folderName='LFP_5C_14-3mg_additive', 
                 save_folder_base='Cycling/plots', overwrite=True)
# 3. Charging Voltage vs Specific Capacity
filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[2, 6], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, legends=True, 
                 points_and_lines=0, column_filter_name='Cycle_Index', 
                 column_X='Specific Capacity (mAh/g)', column_Y='Voltage(V)', 
                 plot_title='Charging Voltage (V) vs Specific Capacity (mAh/g)', 
                 folderName='LFP_5C_14-3mg_additive', save_folder_base='Cycling/plots', overwrite=True)

# 4. Discharging Voltage vs Specific Capacity
filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[3, 7], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, legends=True, 
                 points_and_lines=0, column_filter_name='Cycle_Index', 
                 column_X='Specific Capacity (mAh/g)', column_Y='Voltage(V)', 
                 plot_title='Discharging Voltage (V) vs Specific Capacity (mAh/g)', 
                 folderName='LFP_5C_14-3mg_additive', save_folder_base='Cycling/plots', overwrite=True)

Selected cycles: [3, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1001]


C:\Users\MTP24ME\AppData\Local\Temp\ipykernel_26636\533154170.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling\plots\LFP_5C_14-3mg_additive\Discharge_Retention.png


PNG saved to Cycling\plots\LFP_5C_14-3mg_additive\Discharge_Retention.png
PDF saved to Cycling\plots\LFP_5C_14-3mg_additive\Discharge_Retention.pdf


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling\plots\LFP_5C_14-3mg_additive\Discharge_Retention.pdf
C:\Users\MTP24ME\AppData\Local\Temp\ipykernel_26636\533154170.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: C

PNG saved to Cycling\plots\LFP_5C_14-3mg_additive\Charge_Retention.png
PDF saved to Cycling\plots\LFP_5C_14-3mg_additive\Charge_Retention.pdf


...

PNG saved to Cycling\plots\LFP_5C_14-3mg_additive\Charging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).png
PDF saved to Cycling\plots\LFP_5C_14-3mg_additive\Charging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling\plots\LFP_5C_14-3mg_additive\Charging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf
C:\Users\MTP24ME\AppData\Local\Temp\ipykernel_26636\533154170.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616

PNG saved to Cycling\plots\LFP_5C_14-3mg_additive\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).png
PDF saved to Cycling\plots\LFP_5C_14-3mg_additive\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf


c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:615: PlotnineWarning: Saving 8 x 6 in image.
c:\Users\MTP24ME\AppData\Local\Programs\Python\Python312\Lib\site-packages\plotnine\ggplot.py:616: PlotnineWarning: Filename: Cycling\plots\LFP_5C_14-3mg_additive\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf


('Cycling\\plots\\LFP_5C_14-3mg_additive\\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).png',
 'Cycling\\plots\\LFP_5C_14-3mg_additive\\Discharging_Voltage_(V)_vs_Specific_Capacity_(mAh_g).pdf')